# Sentiment Classification


## Loading the dataset

1. Import Test and Train Data
2. Import Labels

In [0]:
from keras.datasets import imdb

vocab_size = 10000 #vocab size

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size) # vocab_size is no.of words to consider from the dataset, ordering based on frequency.

In [0]:
from keras.preprocessing.sequence import pad_sequences
vocab_size = 10000 #vocab size
maxlen = 300  #number of word used from each review

3.Get the word index and then Create a key-value pair for word and word_id

In [0]:
import keras
import numpy as np

NUM_WORDS=1000 # only use top 1000 words
INDEX_FROM=3   # word index offset

word_to_id = keras.datasets.imdb.get_word_index()
word_to_id = {k:(v+INDEX_FROM) for k,v in word_to_id.items()}
word_to_id["<PAD>"] = 0
word_to_id["<START>"] = 1
word_to_id["<UNK>"] = 2
word_to_id["<UNUSED>"] = 3

id_to_word = {value:key for key,value in word_to_id.items()}



In [77]:
#Pickup a random sample and check the Sentiment
sample_id = np.random.randint(0, x_train.shape[0])

print("sample",sample_id)
print(' '.join(id_to_word[id] for id in x_train[sample_id] ))
print("Sentiment:", y_train[sample_id] )
if(y_train[sample_id]==0):
  print("Negative")
else:
  print("Positive")

sample 18456
<START> this is one of the most god awful movies ever <UNK> better just stick to basketball this movie took away apart of my life i will never have back i will make fun of this movie until i die and then some it is so horrible it is not even funny <UNK> would have a blast with this one
Sentiment: 0
Negative


In [0]:
x_sample = x_test[:500]

In [0]:
#make all sequences of the same length
x_train = pad_sequences(x_train, maxlen=maxlen)
x_test =  pad_sequences(x_test, maxlen=maxlen)

In [0]:
print("train_data ", x_train.shape)
print("train_labels ", y_train.shape)
print("test_data ", x_test.shape)
print("test_labels ", y_test.shape)


In [0]:
x_val = x_train[:10000]
y_val = y_train[:10000]

x_train = x_train[10000:]
y_train = y_train[10000:]

In [0]:
print("train_data ", x_train.shape)
print("train_labels ", y_train.shape)
print("val_data ", x_val.shape)
print("val_labels ", y_val.shape)


train_data  (15000, 300)
train_labels  (15000,)
val_data  (10000, 300)
val_labels  (10000,)


## Build Keras Embedding Layer Model
We can think of the Embedding layer as a dicionary that maps a index assigned to a word to a word vector. This layer is very flexible and can be used in a few ways:

* The embedding layer can be used at the start of a larger deep learning model. 
* Also we could load pre-train word embeddings into the embedding layer when we create our model.
* Use the embedding layer to train our own word2vec models.

The keras embedding layer doesn't require us to onehot encode our words, instead we have to give each word a unqiue intger number as an id. For the imdb dataset we've loaded this has already been done, but if this wasn't the case we could use sklearn [LabelEncoder](http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html).

**4**. **Build a Sequential Model using Keras for the Sentiment Classification task**

In [0]:
import tensorflow as tf

#Initialize model
tf.keras.backend.clear_session()
model = tf.keras.Sequential()

In [7]:
top_words = 10000 #Vocablury size
max_review_length = 300 #maximum number of words to consider in each review

model.add(tf.keras.layers.Embedding(top_words + 1, #Vocablury size + 1 for padding
                                    50, #Embedding size
                                    input_length=max_review_length) #Number of words in each review
          )

Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
If using Keras pass *_constraint arguments to layers.


In [0]:
model.add(tf.keras.layers.LSTM(256, #RNN State - size of cell state and hidden state
                               dropout=0.2, #Dropout before feeding the data to LSTM layer
                               recurrent_dropout=0.2)) #Dropout applied to the output of LSTM layer

In [0]:
model.output

<tf.Tensor 'lstm/strided_slice_7:0' shape=(?, 256) dtype=float32>

In [0]:
model.add(tf.keras.layers.Dense(1,activation='sigmoid'))

In [10]:
#Compile the model
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where


In [0]:
model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding (Embedding)        (None, 300, 50)           500050    
_________________________________________________________________
lstm (LSTM)                  (None, 256)               314368    
_________________________________________________________________
dense (Dense)                (None, 1)                 257       
_________________________________________________________________
dense_1 (Dense)              (None, 1)                 2         
Total params: 814,677
Trainable params: 814,677
Non-trainable params: 0
_________________________________________________________________


In [11]:
from google.colab import drive
drive.mount('/content/drive')

Go to this URL in a browser: https://accounts.google.com/o/oauth2/auth?client_id=947318989803-6bn6qk8qdgf4n4g3pfee6491hc0brc4i.apps.googleusercontent.com&redirect_uri=urn%3aietf%3awg%3aoauth%3a2.0%3aoob&response_type=code&scope=email%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdocs.test%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive.photos.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fpeopleapi.readonly

Enter your authorization code:
··········
Mounted at /content/drive


In [12]:
%cd /content/drive/My\ Drive/AIML/Sequential\ NLP/Project1/

/content/drive/My Drive/AIML/Sequential NLP/Project1


In [0]:
!ls

best_model_NLP.h5  SeqNLP_Project1_Questions.ipynb


In [0]:
from keras.callbacks import ModelCheckpoint,EarlyStopping
filepath = "best_model_NLP.h5"
checkpoint = ModelCheckpoint(filepath, monitor='val_acc', verbose=1, save_best_only=True, mode='max')
cp2 = EarlyStopping(monitor='val_loss', patience=50, verbose=0)
callbacks_list = [checkpoint,cp2]

In [84]:
model.fit(x_train,y_train,
          epochs=5,
          batch_size=32,          
          validation_data=(x_val, y_val),verbose=1,callbacks=callbacks_list)

Train on 15000 samples, validate on 10000 samples
Epoch 1/5
14976/15000 [============================>.] - ETA: 0s - loss: 0.2416 - acc: 0.9070
Epoch 00001: val_acc did not improve from 0.84550
15000/15000 [==============================] - 352s 23ms/sample - loss: 0.2414 - acc: 0.9071 - val_loss: 0.4167 - val_acc: 0.8351
Epoch 2/5
14976/15000 [============================>.] - ETA: 0s - loss: 0.1622 - acc: 0.9426
Epoch 00002: val_acc did not improve from 0.84550
15000/15000 [==============================] - 349s 23ms/sample - loss: 0.1623 - acc: 0.9425 - val_loss: 0.4927 - val_acc: 0.8048
Epoch 3/5
14976/15000 [============================>.] - ETA: 0s - loss: 0.1443 - acc: 0.9489
Epoch 00003: val_acc did not improve from 0.84550
15000/15000 [==============================] - 351s 23ms/sample - loss: 0.1443 - acc: 0.9489 - val_loss: 0.4719 - val_acc: 0.8296
Epoch 4/5
14976/15000 [============================>.] - ETA: 0s - loss: 0.1226 - acc: 0.9569
Epoch 00004: val_acc did not impro

In [14]:
!ls

best_model_NLP.h5  SeqNLP_Project1_Questions.ipynb


In [22]:
# load saved model
load_model_path = 'best_model_NLP.h5'
loaded_model = tf.keras.models.load_model(load_model_path)

Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


**5. Report the Accuracy of the model**

In [23]:
# evaluate
loss, acc = loaded_model.evaluate(x_test, y_test, verbose=0)
print('Test Accuracy: %f' % (acc*100))

Test Accuracy: 83.564001


In [81]:
# Test with a random sample
sample_id = np.random.randint(0, x_sample.shape[0])

print("sample",sample_id)
sample_text = ' '.join(id_to_word[id] for id in x_sample[sample_id] )
print(sample_text)

sample 270
<START> this soft soft core sci fi b movie is what you'd have if you took an early fred <UNK> ray film and took out the fun or <UNK> it's like an uwe boll <UNK> but without as much <UNK> a young <UNK> chain gang convict c c <UNK> agrees to pose as a space <UNK> in order to stop wacky kim <UNK> plans of having everyone have sex with everyone else apparently that vile fiend anyone who went into this film looking for serious science fiction well you got what you deserved for not doing any <UNK> on the film at all first of all when did kim dawson ever star in anything other than soft core <UNK> level crap for that matter take a look at the <UNK> for <UNK> and the director before you take a <UNK> fit saying you expected something else don't get me wrong for a space action soft core <UNK> flick this film is still not good but if you expected something along the lines of contact i do not pity you br br my grade d br br where i saw it <UNK> on demand available until december <UNK> 2

In [83]:
# Predict Sentiment from sample review

from keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer(num_words=maxlen)

instance = tokenizer.texts_to_sequences(sample_text)

flat_list = []
for sublist in instance:
    for item in sublist:
        flat_list.append(item)

flat_list = [flat_list]

instance = pad_sequences(flat_list, padding='post', maxlen=maxlen)

loaded_model.predict(instance)

array([[0.96775436]], dtype=float32)

In [0]:
#we mapped the positive outputs to 1 and the negative outputs to 0. 
#However, the sigmoid function predicts floating value between 0 and 1. 
#If the value is less than 0.5, the sentiment is considered negative where as if the value is greater than 0.5, the sentiment is considered as positive. 

6. Retrieve the output of each layer in Keras for a given single test sample from the trained
model you built

In [0]:
inp = model.input                                    # input placeholder
outputs = [layer.output for layer in model.layers[1:]] # exclude Input    # all layer outputs
functors = [K.function([inp, K.learning_phase()], [out]) for out in outputs]    # evaluation functions


In [0]:
# Testing

input_shape = [1] + list(model.input_shape[1:])
layer_outs = [func([instance, 1.]) for func in functors]
print(layer_outs)
